# RAG

RAG (Retrieval-Augmented Generation) — это продвинутый подход в обработке естественного языка, который сочетает сильные стороны поиска информации и моделей генерации текста.

Обычные LLM, такие как GPT, создают ответы, опираясь на шаблоны, усвоенные из больших объёмов данных, но им может не хватать точной и актуальной информации.
RAG решает эту проблему, сначала извлекая из базы данных релевантные документы или фрагменты текста, а затем используя их для уточнения и улучшения процесса генерации.

Такой гибридный метод обеспечивает, чтобы создаваемый текст был одновременно контекстно уместным и фактически точным.
Благодаря встроенным механизмам поиска модели RAG могут динамически получать и использовать самую свежую информацию, что делает их особенно эффективными для задач вроде ответов на вопросы, поддержки клиентов и создания контента.

In [17]:
import os
import sys

import numpy as np

run_env = os.getenv('RUN_ENV', 'COLLAB')
if run_env == 'COLLAB':
  from google.colab import drive
  ROOT_DIR = '/content/drive'
  drive.mount(ROOT_DIR)
  print('Google drive connected')
  root_data_dir = os.path.join(ROOT_DIR, 'MyDrive', 'ml_course_data')
  lib_path = os.path.join(ROOT_DIR, 'MyDrive', 'src')
  if not os.path.exists(lib_path):
    raise RuntimeError('Upload and `src` dir with code')
  sys.path.append(lib_path)
else:
  repo_dir = os.path.dirname(os.getcwd())  # notebook runs from jupyter_notebooks/
  src_path = os.path.join(repo_dir, 'src')
  if src_path not in sys.path:
    sys.path.append(src_path)
  root_data_dir = os.getenv('DATA_DIR', '/srv/data')

if not os.path.exists(root_data_dir):
  raise RuntimeError('Data dir not exists')
else:
  print('Data dir content %s: %s' % (root_data_dir, ', '.join(os.listdir(root_data_dir)[:5])))

Data dir content /Users/adzhumurat/PycharmProjects/ai_product_engineer/data: scraping, client_segmentation.csv, messages.db, labeled_data_corpus.csv, nebius_site


It is critical to get access for GPU for fast embeddings evaluation

In [18]:
import torch

device = torch.device(
    [f'cuda:{i}' for i in range(torch.cuda.device_count())][0]
    if torch.cuda.is_available() else 'cpu'
)
if not 'cuda' in str(device):
    # raise RuntimeError('GPU is not available')
    print(f'GPU is not available: {str(device)}')
print('torch device: %s' % device)

GPU is not available: cpu
torch device: cpu


In [19]:
models_dir = os.path.join(root_data_dir, 'models')
if not os.path.exists(models_dir):
    os.mkdir(models_dir)
print(f'Models dir: {models_dir}')

Models dir: /Users/adzhumurat/PycharmProjects/ai_product_engineer/data/models


In [20]:
import os

if run_env == 'COLLAB':
    with open(os.path.join(ROOT_DIR, 'MyDrive', 'secrets', 'secrets.env'), 'r') as f:
      envs = {j[0]: j[1] for j in [i.strip().split('=') for i in f.readlines()]}
    print(len(envs))

In [21]:
from IPython.display import clear_output

!pip install langchain_community python-dotenv langchain_huggingface backoff
# !pip install faiss-gpu-cu11
clear_output()
print('Packages installed')

Packages installed


# RAG dev

[amazon reviews](https://amazon-reviews-2023.github.io/)

Plan
* rag over one dataset (healthcare)
* rag over two datasets (healthcare + electronics)

In [22]:
from rag.files_processing import read_raw_data

read_raw_data('Health_and_Personal_Care.jsonl.gz', limit = 3, data_dir=root_data_dir)

Dataset num items: 3


[{'rating': 4.0,
  'title': '12 mg is 12 on the periodic table people! Mg for magnesium',
  'text': 'This review is more to clarify someone else’s review bc they didn’t understand understand the labeling!  It shows 1000mg as advertised & another little label says 12mg bc 12 is on the periodic table for magnesium!  I realize not everyone takes chemistry, but 4 ppl liked his review & so misinformation is spreading.  This works. If however you are on opiate level medications that are causing constipation you should talk to your pain dr or your gastrointestinal dr & ask for a medication called Linzess which works must better & must faster, but is unnecessary for most people.  If magnesium is working for you just make sure to take it with food & drink 6-8 glasses of water per day.  Staying hydrated will really help.  Before switching to Linzess I used to take one 1,000 mg pill am & pm every day with meals & always with an 8 ounce glass of water or other liquid.',
  'images': [],
  'asin': '

In [23]:
import pandas as pd

def count_products_popularity(raw_data_entries):
  db_sample_df = pd.json_normalize(
      raw_data_entries
  )
  popularity_df = (
    db_sample_df
    .groupby('asin')['user_id']
    .count()
    .reset_index(name='popularity')
    .sort_values(by='popularity', ascending=False)
  )
  return popularity_df

count_products_popularity(
    read_raw_data('Health_and_Personal_Care.jsonl.gz', limit=1000, fields = None)
).head()

Task was destroyed but it is pending!
task: <Task pending name='Task-263' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/adzhumurat/PycharmProjects/ai_product_engineer/.venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-264' coro=<Kernel.shell_main() running at /Users/adzhumurat/PycharmProjects/ai_product_engineer/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/adzhumurat/PycharmProjects/ai_product_engineer/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/adzhumurat/.local/share/uv/python/cpython-3.13.5-macos-aarch64-none/lib/python3.13/json/decoder.py:361: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-264' coro=<Kernel.shell_main() running at /Users/adzhumurat/PycharmProjec

Dataset num items: 1000


,asin,popularity
9,B000G2BESO,7
350,B07HSF5HTX,7
343,B07GHG9P9P,4
827,B09F7PH4QR,4
351,B07J32R15F,3


# Huggingface

возможно потребуется авторизация


```shell
hf auth login
```

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    cache_folder=models_dir,
    model_kwargs={"device": device}  # 👈 force GPU, if available
)

corpus_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming industries."
]

# Generate embeddings for documents
passage_embeddings = embedding_model.embed_documents(corpus_texts)

print(f"Generated {len(passage_embeddings)} embeddings")
print(f"Each embedding vector has dimension {len(passage_embeddings[0])}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generated 2 embeddings
Each embedding vector has dimension 768


In [25]:
from rag.files_processing import load_corpus
from rag.embedder import train_embeds


db = read_raw_data(
    'Health_and_Personal_Care.jsonl.gz',
    limit = -1, fields = ['rating', 'text', 'title', 'asin', 'user_id'],
    data_dir=root_data_dir
)
products_popularity_df = count_products_popularity(db)
print(f'Num rows {products_popularity_df.shape[0]}')
products_popularity_df.head()

Dataset num items: 494121
Num rows 62597


,asin,popularity
850,B000G2BESO,2840
4562,B004C7MTLA,2196
312,B0001ZWPI4,2095
6228,B0077L8YFI,1992
15616,B00XQBHJOU,1952


In [26]:
products_filter = set(products_popularity_df.head(10000)['asin'].values)
corpus = load_corpus(db, max_reviews_per_product=4, product_filter=products_filter)
print(f'Corpus length {len(corpus)}')
corpus[0]

Corpus length 49620


{'doc_id': 'B07TDSJZMR',
 'text': 'This review is more to clarify someone else’s review bc they didn’t understand understand the labeling!  It shows 1000mg as advertised & another little label says 12mg bc 12 is on the periodic table for magnesium!  I realize not everyone takes chemistry, but 4 ppl liked his review & so misinformation is spreading.  This works. If however you are on opiate level medications that are causing constipation you should talk to your pain dr or your gastrointestinal dr & ask for a medication called Linzess which works must better & must faster, but is unnecessary for most people.  If magnesium is working for you just make sure to take it with food & drink 6-8 glasses of water per day.  Staying hydrated will really help.  Before switching to Linzess I used to take one 1,000 mg pill am & pm every day with meals & always with an 8 ounce glass of water or other liquid.'}

In [27]:
data_version = 2
embeds = train_embeds(
    [i['text'] for i in corpus], embedding_model,
    os.path.join(root_data_dir, f'corpus_embeds_{data_version}.npy'),
    overwrite=False
)
print(embeds.shape)

corpus loading from /Users/adzhumurat/PycharmProjects/ai_product_engineer/data/corpus_embeds_2.npy
Num embeddings 49620
(49620, 768)


In [29]:
query_string = 'headache'

query_vector = embedding_model.embed_documents([query_string])
len(query_vector[0])
query_vector[0][:10]

[-0.001411128556355834,
 -0.045992571860551834,
 -0.006424997001886368,
 -0.037270110100507736,
 0.01663815602660179,
 0.032171670347452164,
 -0.023516757413744926,
 0.04734068736433983,
 0.04063330218195915,
 0.017658255994319916]

In [30]:
from rag.embedder import VectorIndex

class Catalog:
  def __init__(self, corpus):
      self.texts = np.array([i['text'] for i in corpus])
      self.doc_ids = np.array([i['doc_id'] for i in corpus])

  def get_items(self, item_indices):
      docs = [{'doc_id': str(self.doc_ids[i]), 'text': str(self.texts[i])} for i in item_indices]
      return docs

vector_index = VectorIndex(embedding_model, corpus, corpus_embeddings=embeds)
content_catalog = Catalog(corpus)
user_query = 'headache'
products = vector_index.retrieve(user_query)
print(products)

Index built successfully. ✅
[{'idx': np.int64(43032), 'score': np.float64(0.545)}, {'idx': np.int64(27990), 'score': np.float64(0.5297)}, {'idx': np.int64(17149), 'score': np.float64(0.4861)}, {'idx': np.int64(9732), 'score': np.float64(0.4744)}, {'idx': np.int64(8471), 'score': np.float64(0.4726)}, {'idx': np.int64(30649), 'score': np.float64(0.4462)}, {'idx': np.int64(14362), 'score': np.float64(0.4373)}, {'idx': np.int64(33488), 'score': np.float64(0.4372)}, {'idx': np.int64(23111), 'score': np.float64(0.4312)}, {'idx': np.int64(12445), 'score': np.float64(0.4306)}]


In [31]:
product_ids = [i['idx'] for i in products]
candidates = content_catalog.get_items(product_ids)
for i in candidates:
  print(i)

{'doc_id': 'B076HF85SK', 'text': 'Awful terrible doesn’t help gave me a big headache for over a werk'}
{'doc_id': 'B002C6467S', 'text': "This is the only medicine that cures my headaches it's the best"}
{'doc_id': 'B00890YO0U', 'text': 'Great product for any type of pain including a headache'}
{'doc_id': 'B01GIP0SW0', 'text': 'Stuffed up nose, this works really well to clear you up'}
{'doc_id': 'B074HSHL59', 'text': 'Need somemtype of migraine relief for the Mrs shemloves the spray.'}
{'doc_id': 'B01NAXDO1T', 'text': 'Wow works on my sore neck and headaches since I was badly injured.'}
{'doc_id': 'B07FYKC837', 'text': 'This product really helps me with my headaches.  I highly recommend it.'}
{'doc_id': 'B07JFYBN1C', 'text': 'I swear by these headache wraps. We purchased 2! The pink one & this black one. The pressure & ice felt when using this almost always receives our migraine head pain. This is something we will always have in our home. We tell everyone that has a migraine about this

## Generation

In [15]:
from llm import get_nebius_client, generate

nebius_client = get_nebius_client()

knowledgebase = '\n'.join(["item_id: %s; review: %s" % (i['doc_id'], i['text']) for i in candidates])

system_prompt = f"""
  You are a helpful assistant for medicine shopping. Answer to user query based on provided context

  {knowledgebase}
"""

user_prompt = f"User query: {user_query}"


In [32]:
genai_resp = generate(nebius_client, user_prompt=user_prompt, system_prompt=system_prompt)
print(genai_resp.text)

Based on the provided reviews, here are the products that are specifically mentioned as helpful for headaches:

- **B002C6467S**: Multiple reviews state this is the only medicine that cures headaches and is the best for migraine relief.
- **B01GIP0SW0**: Mentioned as effective for headaches and sore neck pain.
- **B07FYKC837**: The product is said to really help with headaches and is highly recommended.
- **B07JFYBN1C**: Headache wraps (with pressure and ice) are described as highly effective for migraine relief and are a must-have in the home.
- **B074HSHL59**: A spray that provides migraine relief, loved by the user’s wife.

**Note**: Some products like **B09J2VHMHB** (patches) were not effective for migraine relief, and **B076HF85SK** caused a headache, so they are not recommended for headaches.

**Top recommendations for headaches**:
- **B002C6467S** (medicine)
- **B07JFYBN1C** (headache wraps)
- **B074HSHL59** (migraine spray)


# RAG: Classical vs Agentic


Classical RAG: фиксированный pipeline
```
Query → Embedding → Vector Search → Top-K → LLM → Answer

Проблемы:
❌ Один проход — нет возможности исправить ошибки
❌ Всегда ищет, даже если не нужно
❌ Не может адаптировать стратегию
```

### Agentic RAG: Actor-Critic паттерн
```
Query → Agent (Actor) → Decision Loop
           ↓
    Нужен ли retrieval?
           ↓
    [Vector Search / Web / SQL / Direct Answer]
           ↓
    Critic оценивает качество
           ↓
    Хорошо? → Ответ
    Плохо? → Reformulate и повторить
```

**Actor** принимает решения:
- Какой инструмент использовать?
- Сколько документов нужно?
- Нужно ли переформулировать запрос?

**Critic** оценивает:
- Relevance score (насколько релевантны документы)
- Faithfulness (нет ли галлюцинаций)
- Completeness (достаточно ли информации)

**Сравнение:**

| Аспект | Classical RAG | Agentic RAG |
|--------|---------------|-------------|
| Архитектура | Линейный pipeline | Циклический с reasoning |
| Retrieval | Всегда выполняется | Только когда нужно |
| Tools | Только vector search | Multiple (web, SQL, code) |
| Latency | Предсказуемая | Переменная |
| Quality | Зависит от первого retrieval | Адаптивное улучшение |

## References

- Agentic RAG [arxiv-paper-curator](https://github.com/jamwithai/arxiv-paper-curator)
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401) — оригинальная RAG статья
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [LangChain Documentation on Agents](https://python.langchain.com/docs/modules/agents/)


# Homework

Use huggingface for an inference

use [InferenceClient](https://huggingface.co/docs/huggingface_hub/main/en/package_reference/inference_client#huggingface_hub.InferenceClient.chat_completion.example) for free
